# Lab 7.1 — Build RAG over Course-Relevant Corpus

End-to-end walkthrough of the Financial Report Analyst pipeline.

**Corpus:** 9 SEC 10-K filings — MSFT, AAPL, NVDA, last 3 fiscal years each — pinned in `data/filings_manifest.yaml`.

**Stack:** LlamaIndex + `BAAI/bge-small-en-v1.5` embeddings (local, CPU) + GitHub Models `gpt-4o-mini` (switchable to OpenAI).

This notebook **imports from `src/financial_analyst/`** rather than duplicating logic — so any improvements made for Lab 7.2 / 7.3 will reflect here on the next run.

---

## 0. Path setup

Make sure the `src/` package is importable when running the notebook directly with `uv run jupyter`.

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
while ROOT.name != 'financial-report-analyst' and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
print('Project root:', ROOT)

Project root: C:\repos\financial-report-analyst


## 1. Inspect the pinned filings manifest

In [2]:
import pandas as pd
from financial_analyst.manifest import load_manifest

filings = load_manifest()
df = pd.DataFrame([{
    'company': f.company,
    'ticker': f.ticker,
    'fiscal_year': f.fiscal_year,
    'filing_date': f.filing_date,
    'period_of_report': f.period_of_report,
    'accession': f.accession,
    'cached': f.local_path.exists(),
} for f in filings])
df

,company,ticker,fiscal_year,filing_date,period_of_report,accession,cached
0,microsoft,MSFT,2025,2025-07-30,2025-06-30,0000950170-25-100235,True
1,microsoft,MSFT,2024,2024-07-30,2024-06-30,0000950170-24-087843,True
2,microsoft,MSFT,2023,2023-07-27,2023-06-30,0000950170-23-035122,True
3,apple,AAPL,2025,2025-10-31,2025-09-27,0000320193-25-000079,True
4,apple,AAPL,2024,2024-11-01,2024-09-28,0000320193-24-000123,True
5,apple,AAPL,2023,2023-11-03,2023-09-30,0000320193-23-000106,True
6,nvidia,NVDA,2026,2026-02-25,2026-01-25,0001045810-26-000021,True
7,nvidia,NVDA,2025,2025-02-26,2025-01-26,0001045810-25-000023,True
8,nvidia,NVDA,2024,2024-02-21,2024-01-28,0001045810-24-000029,True


## 2. Run the section splitter and review per-filing parse report

The splitter fails loud if it can't find the required sections (Item 1, 1A, 1C*, 7, 7A, 8). \* Item 1C only required for fiscal years ending on/after 2023-12-15.

In [3]:
from financial_analyst.ingest import ingest_filing, required_sections_for, ITEM_ORDER

rows = []
for f in filings:
    r = ingest_filing(f)
    by_id = {s.item_id: s for s in r.sections}
    required = required_sections_for(f.period_of_report)
    rows.append({
        'company': f.company,
        'fy': f.fiscal_year,
        'plain_chars': r.plain_text_chars,
        'sections': len(r.sections),
        'item_1_chars': by_id.get('1').char_count if by_id.get('1') else 0,
        'item_1A_chars': by_id.get('1A').char_count if by_id.get('1A') else 0,
        'item_7_chars': by_id.get('7').char_count if by_id.get('7') else 0,
        'item_8_chars': by_id.get('8').char_count if by_id.get('8') else 0,
        'required_ok': all(by_id.get(rid) for rid in required),
        'errors': '; '.join(r.errors) or '-',
    })
pd.DataFrame(rows)

c:\repos\financial-report-analyst\.venv\Lib\site-packages\pydantic\_internal\_generate_schema.py:2274: UnsupportedFieldAttributeWarning: The 'validate_default' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'validate_default' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(


,company,fy,plain_chars,sections,item_1_chars,item_1A_chars,item_7_chars,item_8_chars,required_ok,errors
0,microsoft,2025,347386,22,29889,68967,47898,100292,True,-
1,microsoft,2024,390411,22,67275,73729,51634,110296,True,-
2,microsoft,2023,373684,21,72849,68531,55637,104754,True,-
3,apple,2025,220552,21,16050,68044,18017,61256,True,-
4,apple,2024,218573,22,15756,68758,15345,60140,True,-
5,apple,2023,216621,21,15151,67875,15488,62262,True,-
6,nvidia,2026,360408,17,48241,114916,34154,206,True,-
7,nvidia,2025,368162,17,54810,110844,37787,206,True,-
8,nvidia,2024,358803,17,51601,106917,40235,206,True,-


## 3. Build (or load) the persisted vector index

In [4]:
from financial_analyst.index import build_index, IndexConfig

cfg = IndexConfig()  # baseline: chunk_size=512, overlap=50
index = build_index(cfg)
print(f'Index loaded with {len(index.docstore.docs)} nodes from {cfg.storage_dir().name}')

Loading llama_index.core.storage.kvstore.simple_kvstore from C:\repos\financial-report-analyst\storage\c512_o50_bge-small-en-v1.5_5c38ec7c_9e02b0f25518\docstore.json.
Loading llama_index.core.storage.kvstore.simple_kvstore from C:\repos\financial-report-analyst\storage\c512_o50_bge-small-en-v1.5_5c38ec7c_9e02b0f25518\index_store.json.
Index loaded with 1601 nodes from c512_o50_bge-small-en-v1.5_5c38ec7c_9e02b0f25518


## 4. Inspect chunk-level metadata

Confirms that company, fiscal_year, section_id, and source_url survived ingestion → chunking.

In [5]:
sample = list(index.docstore.docs.values())[0]
print('Sample chunk metadata:')
for k, v in sample.metadata.items():
    print(f'  {k:>22}: {v}')
print(f'\nText (first 200 chars):\n{sample.get_content()[:200]}...')

# Distribution across (company, fiscal_year, section_id)
from collections import Counter
by_section = Counter((d.metadata['company'], d.metadata['fiscal_year'], d.metadata['section_id'])
                     for d in index.docstore.docs.values())
pd.DataFrame(
    [(c, fy, sid, n) for (c, fy, sid), n in by_section.most_common(15)],
    columns=['company', 'fy', 'section_id', 'n_chunks'],
)

Sample chunk metadata:
                 company: microsoft
                  ticker: MSFT
             fiscal_year: 2025
             filing_date: 2025-07-30
        period_of_report: 2025-06-30
             report_type: 10-K
               accession: 0000950170-25-100235
              source_url: https://www.sec.gov/Archives/edgar/data/789019/000095017025100235/msft-20250630.htm
              section_id: item_1
           section_title: Business
      section_char_count: 29889

Text (first 200 chars):
Item 1 Our database, business intelligence, and data warehousing solutions offerings compete with products from providers in the data and analytics industry. Our system management solutions compete wi...


,company,fy,section_id,n_chunks
0,nvidia,2025,item_11,76
1,microsoft,2024,item_8,75
2,nvidia,2024,item_11,75
3,nvidia,2026,item_11,74
4,microsoft,2025,item_8,69
5,microsoft,2023,item_8,69
6,nvidia,2026,item_1a,57
7,nvidia,2025,item_1a,57
8,nvidia,2024,item_1a,55
9,apple,2023,item_8,46


## 5. Three sample queries with citations

Each query is run through the same `answer_with_citations` flow exposed by `scripts/run_query.py`.

In [6]:
from financial_analyst.retrieve import build_retriever, infer_filters_from_question
from financial_analyst.synthesize import answer_with_citations

questions = [
    "How did NVIDIA's data center revenue change from FY2023 to FY2024?",
    "What are the most prominent business risks Apple highlighted in its FY2024 filing?",
    "Compare Microsoft's and NVIDIA's commentary on AI demand.",
]
for q in questions:
    flt = infer_filters_from_question(q).to_llama_filters()
    r = build_retriever(index, similarity_top_k=5, filters=flt)
    ans = answer_with_citations(r, q)
    print('=' * 80)
    print('Q:', q)
    print()
    print(ans.render())
    print()

Q: How did NVIDIA's data center revenue change from FY2023 to FY2024?

NVIDIA’s Data Center revenue increased from $15.1 billion in FY2023 to $47.5 billion in FY2024, a rise of 217%. [2, 4]

Sources:
  [1] NVIDIA FY2024 — Item 7 (Management's Discussion and Analysis)  (score=0.830)
      We did not experience any significant impact or expense to our business; however, if the conflict is further extended, it could impact future product development, operations, and revenue or create other uncertainty for o...
  [2] NVIDIA FY2024 — Item 7 (Management's Discussion and Analysis)  (score=0.820)
      Gaming revenue for fiscal year 2024 was up 15%. The increase reflects higher sell-in to partners following the normalization of channel inventory levels and growing demand. Professional Visualization revenue for fiscal y...
  [3] NVIDIA FY2024 — Item 7 (Management's Discussion and Analysis)  (score=0.815)
      Our estimated Compute & Networking demand is expected to remain concentrated. There w

## 6. Sanity check: reload the persisted index

Confirms that the docstore + index_store + vector_store all round-trip cleanly.

In [7]:
from financial_analyst.index import verify_reload
verify_reload(cfg)

Loading llama_index.core.storage.kvstore.simple_kvstore from C:\repos\financial-report-analyst\storage\c512_o50_bge-small-en-v1.5_5c38ec7c_9e02b0f25518\docstore.json.
Loading llama_index.core.storage.kvstore.simple_kvstore from C:\repos\financial-report-analyst\storage\c512_o50_bge-small-en-v1.5_5c38ec7c_9e02b0f25518\index_store.json.
Loading llama_index.core.storage.kvstore.simple_kvstore from C:\repos\financial-report-analyst\storage\c512_o50_bge-small-en-v1.5_5c38ec7c_9e02b0f25518\docstore.json.
Loading llama_index.core.storage.kvstore.simple_kvstore from C:\repos\financial-report-analyst\storage\c512_o50_bge-small-en-v1.5_5c38ec7c_9e02b0f25518\index_store.json.


True